In [ ]:

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings("ignore")


##  Kaggle Dataset Import & Setup


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("atharvaingle/crop-recommendation-dataset")

print("Path to dataset files:", path)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("abhinand05/crop-production-in-india")

print("Path to dataset files:", path)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("anishaman07/agmarknet-india-commodity-prices-oct24-aug25")

print("Path to dataset files:", path)

In [ ]:
import os

print("Soil & Climate files:")
print(os.listdir("/kaggle/input/crop-recommendation-dataset"))

print("\nCrop Production files:")
print(os.listdir("/root/.cache/kagglehub/datasets/abhinand05/crop-production-in-india/versions/1"))

print("\nAgmarknet Price files:")
print(os.listdir("/kaggle/input/agmarknet-india-commodity-prices-oct24-aug25"))


In [ ]:
import os

base_path = "/kaggle/input/agmarknet-india-commodity-prices-oct24-aug25"

print(os.listdir(base_path))

subfolder = base_path + "/agmarknet-india-commodity-prices-2024-2025"
print(os.listdir(subfolder))


# 1. Soil & Climate Dataset Processing (Agro-Zoning Input)



In [ ]:
soil_df = pd.read_csv("/kaggle/input/crop-recommendation-dataset/Crop_recommendation.csv")

soil_df.head()


In [ ]:
soil_df.info()

In [ ]:
# Remove label
# Since this project focuses on UNSUPERVISED agro-zoning,
# the label must be removed to avoid supervised learning bias.

soil_X = soil_df.drop(columns=["label"])
soil_X.head()

In [ ]:
soil_X.describe()

In [ ]:
soil_X.isnull().sum()

# 2. Crop Production Dataset Processing (Historical Yield Analysis)


In [ ]:
crop_prod_df = pd.read_csv("/root/.cache/kagglehub/datasets/abhinand05/crop-production-in-india/versions/1/crop_production.csv")

crop_prod_df.head()


In [ ]:
crop_prod_df.info()

In [ ]:
# -------------------------------
# Step 2.1: Check missing values
# -------------------------------
# Production is required to compute yield.
# Rows with missing production cannot contribute to historical performance.

crop_prod_df.isnull().sum()


In [ ]:
# Yield computation requires production data.
# Rows with missing production are removed to ensure valid yield estimation.

crop_prod_df = crop_prod_df.dropna(subset=["Production"])

crop_prod_df.isnull().sum()


In [ ]:
# -------------------------------
# Step 2.2: Compute crop yield
# -------------------------------
# Yield is calculated as Production per unit Area.
# This metric represents historical crop performance independent of land size.

crop_prod_df["Yield"] = crop_prod_df["Production"] / crop_prod_df["Area"]
crop_prod_df.head()

In [ ]:
# check Yield values
crop_prod_df["Yield"].describe()


In [ ]:
# -------------------------------
# Step 2.3: Cap extreme Yield values using IQR
# -------------------------------
# This limits the influence of extreme outliers while
# preserving the overall yield distribution.

Q1 = crop_prod_df["Yield"].quantile(0.25)
Q3 = crop_prod_df["Yield"].quantile(0.75)
IQR = Q3 - Q1

upper_bound = Q3 + 1.5 * IQR

# Cap yield values above the upper bound
crop_prod_df["Yield"] = crop_prod_df["Yield"].clip(upper=upper_bound)


In [ ]:
crop_prod_df["Yield"].describe()


In [ ]:
crop_prod_df['Season'].value_counts()

In [ ]:
# -------------------------------
# Step 2.4 : Clean season names and keep required seasons
# -------------------------------
crop_prod_df["Season"] = crop_prod_df["Season"].str.strip()

allowed_seasons = ["Kharif", "Rabi", "Summer"]
crop_prod_df = crop_prod_df[crop_prod_df["Season"].isin(allowed_seasons)]

crop_prod_df["Season"].value_counts()


In [ ]:
# -------------------------------
# Step 2.5 : Aggregate historical yield
# -------------------------------
# We group data by State, Season, and Crop
# and calculate average yield.
# This gives stable historical performance for each crop.

yield_agg_df = (
    crop_prod_df
    .groupby(["State_Name", "Season", "Crop"], as_index=False)
    .agg({"Yield": "mean"})
)

yield_agg_df.shape

In [ ]:
yield_agg_df.head()

# 3. Market Price Dataset Processing (Economic Value Indicator)


In [ ]:
price_df = pd.read_csv("/kaggle/input/agmarknet-india-commodity-prices-oct24-aug25/"
    "agmarknet-india-commodity-prices-2024-2025/""agmarknet_india_historical_prices_2024_2025.csv")

price_df.head()


In [ ]:
price_df.info()

In [ ]:
# -------------------------------
# Step 3.1: Select required columns from price data
# -------------------------------
# Only crop name and modal price are needed
# for economic comparison in recommendations.

price_df = price_df[["Commodity", "Modal Price (Rs./Quintal)"]]
price_df.head()

In [ ]:
#  Rename columns for clarity

price_df = price_df.rename(columns={
    "Commodity": "Crop",
    "Modal Price (Rs./Quintal)": "Modal_Price"
})
price_df.head()


In [ ]:
# Check missing values in price data
price_df.isnull().sum()


In [ ]:
# -------------------------------
# Step 3.2: Aggregate market price by crop
# -------------------------------
# Multiple price records exist for each crop.
# We take the average modal price to represent
# the overall selling value of that crop.

price_agg_df = (
    price_df
    .groupby("Crop", as_index=False)
    .agg({"Modal_Price": "mean"})
)
price_agg_df.head()

In [ ]:
price_agg_df.shape

In [ ]:
# Rename aggregated price column

price_agg_df = price_agg_df.rename(columns={"Modal_Price": "Avg_Modal_Price"})
price_agg_df.head()

# 4. Agro-Zone Discovery (Unsupervised Clustering)

In [ ]:
# -------------------------------
# Step 4.1: Standardize soil & climate features
# -------------------------------
# K-Means is distance-based.
# Scaling ensures all features contribute fairly.

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

soil_scaled = scaler.fit_transform(soil_X)



In [ ]:
# -------------------------------
# Step 4.2: Elbow Method to find optimal K
# -------------------------------
# We calculate inertia (within-cluster distance)
# for different values of K and observe the elbow point.

from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

inertia = []
K_range = range(2, 11)  # testing K from 2 to 10

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(soil_scaled)
    inertia.append(kmeans.inertia_)

# Plot Elbow Curve
plt.figure()
plt.plot(K_range, inertia, marker='o')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Optimal K")
plt.show()


In [ ]:
# -------------------------------
# Step 4.3: Silhouette score validation for K = 5
# -------------------------------
# After observing the elbow graph, K = 5 looks like a good choice.
# To confirm this, we use the silhouette score.
#
# The silhouette score tells us how well the data points fit within
# their assigned clusters compared to other clusters.
# A higher value means better separation between clusters.

from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans

kmeans_5 = KMeans(n_clusters=5, random_state=42, n_init=10)

cluster_labels = kmeans_5.fit_predict(soil_scaled)

sil_score = silhouette_score(soil_scaled, cluster_labels)

sil_score


In [ ]:
# -------------------------------
# Step 4.4: Fit final K-Means model and assign Zone IDs
# -------------------------------
# Using K = 5 as decided from elbow and silhouette analysis.
# Each soil sample is assigned to one agro-zone.

from sklearn.cluster import KMeans

final_kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)

soil_X["Zone_ID"] = final_kmeans.fit_predict(soil_scaled)


In [ ]:
soil_X["Zone_ID"].value_counts()



In [ ]:
# -------------------------------
# Step 4.6: Visualize agro-zones using PCA (for visualization only)
# -------------------------------
# PCA is used only to reduce dimensions for plotting.
# Clustering is already done on full feature space.

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
soil_pca = pca.fit_transform(soil_scaled)

plt.figure(figsize=(7, 5))
plt.scatter(
    soil_pca[:, 0],
    soil_pca[:, 1],
    c=soil_X["Zone_ID"],
    cmap="tab10",
    s=20
)
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.title("Agro-Zone Clusters (PCA Visualization)")
plt.colorbar(label="Zone_ID")
plt.show()


In [ ]:
# -------------------------------
# Step 4.7: Agro-zone profiling
# -------------------------------
# This step helps us understand the characteristics of each agro-zone by looking at average soil and climate values within each zone.

zone_profile_df = (
    soil_X
    .groupby("Zone_ID")
    .mean()
    .reset_index()
)

zone_profile_df


# Agro-Zone Interpretation (Based on Soil & Climate Profiling)

After clustering the soil and climate data into five agro-zones, we analyzed the average soil nutrients and climatic conditions within each zone.
This profiling helps us understand what type of agricultural land each zone represents and makes the clustering results interpretable.
<br><br>


# 🌱 Zone 0: Low Fertility, Moderate Climate Zone

Low Nitrogen (N) and Potassium (K) levels

Moderate temperature

Lower rainfall compared to other zones

**Interpretation:**
This zone represents relatively dry land with moderate soil fertility. Crops suited to low-input or drought-tolerant conditions may perform better here.

<br>


# 🌾 Zone 1: Nitrogen-Rich, Warm Zone

Very high Nitrogen (N) content

Moderate rainfall

Warm temperature conditions

**Interpretation:**
This zone represents fertile agricultural land with good nitrogen availability, suitable for crops that require higher nutrient levels and warmer climates.

<br>


# 🌧️ Zone 2: High Rainfall, High Humidity Zone

Moderate nutrient levels

Very high rainfall

High humidity

**Interpretation:**
This zone represents wet, rain-fed agricultural regions. Crops that thrive in high moisture and humid conditions are more suitable for this zone.

<br>


# 🧪 Zone 3: Mineral-Rich Soil Zone

Extremely high Potassium (K)

High Phosphorus (P)

Moderate rainfall and temperature

**Interpretation:**
This zone represents mineral-rich soils, which are beneficial for crops that require strong root development and higher nutrient absorption.

<br>



# ❄️ Zone 4: Cool and Dry Zone

Lower temperature

Low humidity

Moderate nutrient levels

**Interpretation:**
This zone represents cooler and relatively dry climatic conditions. Crops adapted to cooler environments and lower moisture levels are more appropriate here.

<br>


# ✅ Summary

These agro-zones provide a meaningful grouping of land based on soil fertility and climate similarity.
In the next phase, these zones will be used to recommend suitable crops for different seasons using historical yield data and market prices.

# 5. Crop Recommendation Logic (Zone-wise & Season-wise)


In [ ]:
# -------------------------------
# Step 5.1: Prepare yield data for recommendations
# -------------------------------
# Since agro-zones are not tied to specific locations,
# we aggregate yield at Season + Crop level.
# This represents general historical performance.

season_crop_yield_df = (
    yield_agg_df
    .groupby(["Season", "Crop"], as_index=False)
    .agg({"Yield": "mean"})
)

season_crop_yield_df.head()


In [ ]:
# -------------------------------
# Step 5.2: Merge yield and market price data
# -------------------------------
# This combines historical crop performance (Yield)
# with economic value (Avg_Modal_Price).

yield_price_df = (
    season_crop_yield_df
    .merge(price_agg_df, on="Crop", how="inner")
)

yield_price_df.head()


In [ ]:
# -------------------------------
# Step 5.3: Normalize Yield and Market Price
# -------------------------------
# Yield is normalized within each season.
# Price is normalized globally across all crops.

from sklearn.preprocessing import MinMaxScaler

# Normalize Yield (season-wise)
yield_price_df["Norm_Yield"] = (
    yield_price_df
    .groupby("Season")["Yield"]
    .transform(lambda x: MinMaxScaler().fit_transform(x.values.reshape(-1, 1)).flatten())
)

# Normalize Price (global)
price_scaler = MinMaxScaler()
yield_price_df["Norm_Price"] = price_scaler.fit_transform(
    yield_price_df[["Avg_Modal_Price"]]
)

yield_price_df.head()


In [ ]:
# -------------------------------
# Step 5.4: Compute recommendation score
# -------------------------------
# The recommendation score combines crop performance and market value.
#
# Yield is given higher importance (0.6) because a crop must grow well
# in a given season to be suitable.
# Market price is given slightly lower importance (0.4) and is used
# to differentiate between crops with similar yield.
#
# These weights are fixed by design to keep the system as a
# decision-support tool, not a profit-optimization model.

yield_price_df["Score"] = (
    0.6 * yield_price_df["Norm_Yield"]
    + 0.4 * yield_price_df["Norm_Price"]
)

yield_price_df.head()


In [ ]:
# -------------------------------
# Step 5.5: Rank crops by score within each season
# -------------------------------
# Crops are ranked in descending order of score
# for each agricultural season.

yield_price_df["Rank"] = (
    yield_price_df
    .groupby("Season")["Score"]
    .rank(method="first", ascending=False)
)

yield_price_df.sort_values(["Season", "Rank"]).head()


In [ ]:
# -------------------------------
# Step 5.6: Select top-3 crops per season
# -------------------------------
# We keep only the top 3 ranked crops
# for each agricultural season.

top3_season_df = (
    yield_price_df
    [yield_price_df["Rank"] <= 3]
    .sort_values(["Season", "Rank"])
)

top3_season_df


### Zone–Season Base Structure


In [ ]:
# -------------------------------
# Zone–Season base structure
# -------------------------------
# This creates all combinations of agro-zones and seasons.
# It is used to attach crop recommendations to each zone.

zones = soil_X["Zone_ID"].unique()
seasons = ["Kharif", "Rabi", "Summer"]

zone_season_df = pd.DataFrame(
    [(z, s) for z in zones for s in seasons],
    columns=["Zone_ID", "Season"]
)

zone_season_df.head()


In [ ]:
# -------------------------------
# Step 5.7: Create zone-wise, season-wise recommendations
# -------------------------------
# Each agro-zone receives the top-3 crops
# for each season based on historical yield and price.

final_recommendation_df = (
    zone_season_df
    .merge(top3_season_df, on="Season", how="left")
    .sort_values(["Zone_ID", "Season", "Rank"])
)

final_recommendation_df.head()


In [ ]:
# -------------------------------
# Step 5.8: Final recommendation output formatting
# -------------------------------
# Select only important columns for final presentation.

final_output_df = final_recommendation_df[
    [
        "Zone_ID",
        "Season",
        "Rank",
        "Crop",
        "Yield",
        "Avg_Modal_Price",
        "Score"
    ]
].reset_index(drop=True)

final_output_df.head()


# 6. Interactive Recommendation Based on User Input


In [ ]:
# -------------------------------
# Step 6.1: Map user soil input to agro-zone
# -------------------------------
# [N, P, K, temperature, humidity, ph, rainfall]

import numpy as np

def get_zone_from_input(N, P, K, temperature, humidity, ph, rainfall):
    input_data = np.array([[N, P, K, temperature, humidity, ph, rainfall]])
    input_scaled = scaler.transform(input_data)
    zone = final_kmeans.predict(input_scaled)[0]
    return zone


## User Input Ranges
### 🌿 Soil Nutrients

* N, P, K: 0 – 140
<BR>


### 🌡️ Climate

* Temperature (°C): 0 – 50
* Humidity (%): 0 – 100
* Rainfall (mm): 0 – 3000
<BR>


### 🧪 Soil pH

* pH: 3.0 – 10.0
<BR>


###🌾 Season

* Kharif, Rabi, Summer
<BR>



---

<BR>


```
  Zone 0 → Low nutrients, low rainfall

  Zone 1 → High Nitrogen, warm

  Zone 2 → High rainfall & humidity

  Zone 3 → Very high K & P (mineral-rich)

  Zone 4 → Cooler, drier climate
```




In [ ]:
# -------------------------------
# Step 6.2: Interactive recommendation interface
# -------------------------------

import ipywidgets as widgets
from IPython.display import display, clear_output

# Bounded inputs with realistic ranges
N_in = widgets.BoundedFloatText(value=50, min=0, max=140, step=1, description="N")
P_in = widgets.BoundedFloatText(value=40, min=0, max=140, step=1, description="P")
K_in = widgets.BoundedFloatText(value=40, min=0, max=140, step=1, description="K")

temp_in = widgets.BoundedFloatText(value=25, min=0, max=50, step=0.5, description="Temp (°C)")
hum_in = widgets.BoundedFloatText(value=70, min=0, max=100, step=1, description="Humidity (%)")
ph_in = widgets.BoundedFloatText(value=6.5, min=3.0, max=10.0, step=0.1, description="pH")
rain_in = widgets.BoundedFloatText(value=150, min=0, max=3000, step=10, description="Rainfall (mm)")

season_dd = widgets.Dropdown(
    options=["Kharif", "Rabi", "Summer"],
    value="Kharif",
    description="Season"
)

button = widgets.Button(description="Get Recommendations", button_style="success")
output = widgets.Output()

def on_button_click(b):
    with output:
        clear_output()

        zone = get_zone_from_input(
            N_in.value,
            P_in.value,
            K_in.value,
            temp_in.value,
            hum_in.value,
            ph_in.value,
            rain_in.value
        )

        print(f"Identified Agro-Zone: {zone}\n")

        display(
            final_output_df[
                (final_output_df["Zone_ID"] == zone) &
                (final_output_df["Season"] == season_dd.value)
            ]
        )

button.on_click(on_button_click)

display(
    N_in, P_in, K_in,
    temp_in, hum_in, ph_in, rain_in,
    season_dd,
    button,
    output
)
